# Module event types from MQTT state messages

This notebook scans the JSONL process logs in `default_process` and lists the action event types reported by each module. An event type is derived from an action's `command` and `state` in a module `/state` payload, for example `MODULE_MILL_FINISHED`. Results are grouped by serial number because the logged state payloads do not include the module class.

The literal module `/status` topic is treated separately: in the available logs it reports connection state (such as `ONLINE`), not action events. The event list below is therefore extracted from `/state` messages.

In [5]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterator


def find_workspace_root() -> Path:
    """Find the repository root from the current notebook working directory."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "default_process").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root (expected a default_process folder) "
        f"above the current directory: {start}"
    )


WORKSPACE_ROOT = find_workspace_root()
LOG_DIR = WORKSPACE_ROOT / "default_process"
JSONL_FILES = sorted(LOG_DIR.glob("*.jsonl"))

if not JSONL_FILES:
    raise FileNotFoundError(f"No JSONL process logs found in {LOG_DIR}")

MODULE_TOPIC_RE = re.compile(
    r"^module/v1/ff/(?:NodeRed/)?(?P<serial>[^/]+)/(?P<kind>state|status)$"
)

print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Process logs: {len(JSONL_FILES)}")
print("\n".join(f"- {path.name}" for path in JSONL_FILES))

Workspace: C:\Users\olive\Documents\Uni\Forschungsprojekt
Process logs: 5
- Ablauf1 100626.jsonl
- Ablauf2 100626.jsonl
- Ablauf2.1 08072026.jsonl
- Ablauf3 100626.jsonl
- Ablauf4 100626.jsonl


In [6]:
def action_states(payload: dict[str, Any]) -> Iterator[dict[str, Any]]:
    """Yield action state objects from either supported payload field."""
    for key in ("actionState", "actionStates"):
        value = payload.get(key)
        if isinstance(value, dict):
            yield value
        elif isinstance(value, list):
            yield from (item for item in value if isinstance(item, dict))


event_rows: list[dict[str, Any]] = []
connection_rows: list[dict[str, Any]] = []

for log_path in JSONL_FILES:
    with log_path.open("r", encoding="utf-8") as log_file:
        for line_number, line in enumerate(log_file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {log_path.name} at line {line_number}: {error}"
                ) from error

            if not isinstance(record, dict):
                continue
            match = MODULE_TOPIC_RE.fullmatch(str(record.get("topic", "")))
            if match is None:
                continue

            payload = record.get("payload")
            if not isinstance(payload, dict):
                continue

            serial = str(payload.get("serialNumber") or match.group("serial"))
            if match.group("kind") == "status":
                connection_rows.append(
                    {
                        "module_serial": serial,
                        "connection_state": payload.get("connectionState"),
                        "source_log": log_path.name,
                        "line_number": line_number,
                    }
                )
                continue

            seen_actions: set[tuple[Any, ...]] = set()
            for action in action_states(payload):
                command = action.get("command")
                if command in (None, ""):
                    continue
                command = str(command).strip().upper()
                state = str(action.get("state") or "UNKNOWN").strip().upper()
                signature = (action.get("id"), command, state)
                if signature in seen_actions:
                    continue
                seen_actions.add(signature)
                event_rows.append(
                    {
                        "module_serial": serial,
                        "command": command,
                        "state": state,
                        "event_type": f"MODULE_{command}_{state}",
                        "source_log": log_path.name,
                        "line_number": line_number,
                    }
                )

if not event_rows:
    raise ValueError(
        f"No module action events found in /state payloads under {LOG_DIR}"
    )

print(f"Action event observations: {len(event_rows):,}")
print(f"Module /status observations: {len(connection_rows):,}")

Action event observations: 1,105
Module /status observations: 7


In [7]:
event_type_stats: dict[tuple[str, str], dict[str, Any]] = {}
for row in event_rows:
    key = (row["module_serial"], row["event_type"])
    stats = event_type_stats.setdefault(
        key, {"message_count": 0, "source_logs": set()}
    )
    stats["message_count"] += 1
    stats["source_logs"].add(row["source_log"])

print("Event types observed for each module serial:")
for module_serial in sorted({row["module_serial"] for row in event_rows}):
    event_types = sorted(
        event_type for serial, event_type in event_type_stats if serial == module_serial
    )
    print()
    print(f"{module_serial}:")
    for event_type in event_types:
        stats = event_type_stats[(module_serial, event_type)]
        source_logs = ", ".join(sorted(stats["source_logs"]))
        print(
            f"  {event_type:<32} {stats['message_count']:>5} observations  "
            f"({len(stats['source_logs'])} logs: {source_logs})"
        )

if connection_rows:
    connection_counts: dict[tuple[str, str], int] = {}
    for row in connection_rows:
        state = str(row["connection_state"] or "UNKNOWN")
        key = (row["module_serial"], state)
        connection_counts[key] = connection_counts.get(key, 0) + 1

    print()
    print("Connection states from literal /status topics (not action event types):")
    for (module_serial, state), count in sorted(connection_counts.items()):
        print(f"  {module_serial:<14} {state:<12} {count:>5} observations")
else:
    print()
    print("No literal module /status topic records were found.")

Event types observed for each module serial:

SVR3QL1089:
  MODULE_DRILL_FINISHED               21 observations  (5 logs: Ablauf1 100626.jsonl, Ablauf2 100626.jsonl, Ablauf2.1 08072026.jsonl, Ablauf3 100626.jsonl, Ablauf4 100626.jsonl)
  MODULE_DRILL_RUNNING                21 observations  (5 logs: Ablauf1 100626.jsonl, Ablauf2 100626.jsonl, Ablauf2.1 08072026.jsonl, Ablauf3 100626.jsonl, Ablauf4 100626.jsonl)
  MODULE_DROP_FINISHED                22 observations  (5 logs: Ablauf1 100626.jsonl, Ablauf2 100626.jsonl, Ablauf2.1 08072026.jsonl, Ablauf3 100626.jsonl, Ablauf4 100626.jsonl)
  MODULE_DROP_RUNNING                 21 observations  (5 logs: Ablauf1 100626.jsonl, Ablauf2 100626.jsonl, Ablauf2.1 08072026.jsonl, Ablauf3 100626.jsonl, Ablauf4 100626.jsonl)
  MODULE_FACTSHEETREQUEST_FINISHED     1 observations  (1 logs: Ablauf1 100626.jsonl)
  MODULE_PICK_FINISHED                21 observations  (5 logs: Ablauf1 100626.jsonl, Ablauf2 100626.jsonl, Ablauf2.1 08072026.jsonl, Ablauf3 10

# Which event types to use for event log from state messages (whitelist)

## SVR3QL1089 (Drill)

MODULE_DRILL_FINISHED

MODULE_DROP_FINISHED

MODULE_PICK_FINISHED

## SVR4H73901 (HBW)

MODULE_DROP_FINISHED 

MODULE_PICK_FINISHED

## SVR4H77011 (AIQS)

MODULE_CHECK_QUALITY_FINISHED

MODULE_DROP_FINISHED

MODULE_PICK_FINISHED

## SVR4H88474 (Mill)

MODULE_DROP_FINISHED

MODULE_MILL_FINISHED

MODULE_PICK_FINISHED

## SVR4H92774 (DPS)

MODULE_DROP_FINISHED

MODULE_INPUT_RGB_FINISHED

MODULE_PICK_FINISHED

MODULE_RGB_NFC_FINISHED

MODULE_SETSTATUSLED_FINISHED